***Loading Packages***

In [2]:
#loading packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import networkx as nx
import seaborn as sns
import igraph as ig
from geopy.distance import geodesic
from scipy.stats import pearsonr
from pathlib import Path
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import sys
import re

project_root = Path.cwd().parent

sys.path.append(str(project_root / "external"))
sys.path.append(str(project_root / "src"))

from real_data_pipeline import (build_baboon_edge_graph, run_baseline_comparison,
                                 run_gnar_edge_rolling_multicov, edge_weights_distribution,
                                 run_full_sweep, build_stages_per_lag)
from simulation import (
    # Network generation
    calibrate_rdp_radius,
    get_or_calibrate_rdp_radius,
    generate_network,
    simulate_node_covariates,
    build_synthetic_edge_covariates,

    # Time-series simulation
    simulate_ar1_exog,
    simulate_full_model,

    # Graph and design-matrix utilities
    build_W_matrices,
    build_design_matrix,
    compute_ols_standard_errors,

    # Monte Carlo simulation
    run_full_simulation_replication,
    run_large_network_replication_density,

    # Regime construction
    expand_R_by_l,
    build_beta_dict,
    build_regime,

    # Results tables
    build_per_parameter_table,
    build_aggregated_table4_style,

    # Small helper
    format_tuple_list,
    format_as_multiindex
)

from diagnostics import save_table, save_figure
from BaseEdgeGNAR_edge import GNAREdgeLearner
from BaseEdgeGNAR_edge_global import GNAREdgeGlobalLearner
from BaseEdge import EdgeTrainSet
from edge_graph import ArrayEdgeGraph
from RollingEdgePredict import RollingEdgePredict
from model import build_edge_covariate, GNAREdgeGlobalMultiCovLearner

#loading external data
interactions_path = project_root / "data" / "baboons_proximity_data.txt"
max_temp_path = project_root / "data" / "trets.csv"
baboon_data_path = project_root / "data" / "baboon_data.csv"

***Parameter Estimation Accuracy for Moderately Sized Networks***

In [ ]:
#calibrate radii for the two target densities (0.1, 0.4), at K=20
radius_01, achieved_01 = calibrate_rdp_radius(K=20, target_density=0.1)
radius_04, achieved_04 = calibrate_rdp_radius(K=20, target_density=0.4)

print(f"density=0.1: radius={radius_01:.3f}, achieved={achieved_01:.3f}")
print(f"density=0.4: radius={radius_04:.3f}, achieved={achieved_04:.3f}")

#storing target radii for rdp graphs
_rdp_radius_cache = {}

#defining simulation regimes
paper_regimes = {
    # gnar-edge paper regimes
    1: {"L": 1, "R_max": [1],       "alpha": [0.2],             "beta_vals": [0.3],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.0, "interaction_pairs": [], "gamma_interaction": {}},
    2: {"L": 1, "R_max": [2],       "alpha": [0.2],             "beta_vals": [(0.3, 0.4)],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.0, "interaction_pairs": [], "gamma_interaction": {}},
    3: {"L": 3, "R_max": [1, 1, 1], "alpha": [0.2, 0.4, -0.6],  "beta_vals": [0.2, 0.1, -0.2],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.0, "interaction_pairs": [], "gamma_interaction": {}},
    4: {"L": 3, "R_max": [2, 2, 2], "alpha": [0.2, 0.4, -0.6],  "beta_vals": [(0.3, 0.1), (0.1, 0.1), (-0.2, 0.3)],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.0, "interaction_pairs": [], "gamma_interaction": {}},
    5: {"L": 3, "R_max": [2, 0, 0], "alpha": [0.2, 0.4, -0.6],  "beta_vals": [(0.3, 0.4)],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.0, "interaction_pairs": [], "gamma_interaction": {}},

    # new regimes with covariates, exogenous series, and/or interactions
    6: {"L": 3, "R_max": [2, 2, 2],
        "alpha": [0.2, 0.4, -0.6],
        "beta_vals": [(0.3, 0.1), (0.1, 0.1), (-0.2, 0.3)],
        "gamma_edge_cov": {"age_diff": 0.05, "mean_age": 0.03}, "gamma_time_cov": 0.0,
        "interaction_pairs": [], "gamma_interaction": {}},
    7: {"L": 3, "R_max": [2, 2, 2],
        "alpha": [0.2, 0.4, -0.6],
        "beta_vals": [(0.3, 0.1), (0.1, 0.1), (-0.2, 0.3)],
        "gamma_edge_cov": {}, "gamma_time_cov": 0.2,
        "interaction_pairs": [], "gamma_interaction": {}},
    8: {"L": 3, "R_max": [2, 2, 2],
        "alpha": [0.2, 0.4, -0.6],
        "beta_vals": [(0.3, 0.1), (0.1, 0.1), (-0.2, 0.3)],   #matches regime 4 exactly
        "gamma_edge_cov": {"age_diff": 0.05, "mean_age": 0.03}, "gamma_time_cov": 0.2,
        "interaction_pairs": [], "gamma_interaction": {}},
    9: {"L": 3, "R_max": [2, 2, 2],
        "alpha": [0.2, 0.4, -0.6],
        "beta_vals": [(0.3, 0.1), (0.1, 0.1), (-0.2, 0.3)],   #matches regime 4 exactly
        "gamma_edge_cov": {"age_diff": 0.05, "mean_age": 0.03}, "gamma_time_cov": 0.2,
        "interaction_pairs": [("mean_age", "sim_temp")], "gamma_interaction": {("mean_age", "sim_temp"): 0.02}},
}



#length of time series, and number of replications
T_values = [200]
n_replications = 50

#converting regime specification into format expected by simulation functions
all_sim_results = []
for regime_id, regime_spec in paper_regimes.items():
    R_by_l, alpha, beta_dict, gamma_edge_cov, gamma_time_cov, interaction_pairs, gamma_interaction = build_regime(regime_spec)
    #looping over structures and repeating n_replications times
    for structure in ["ER", "SBM", "RDP"]:
        for T in T_values:
            for rep in range(n_replications):
                result = run_full_simulation_replication(
                    K=20, density=0.4, structure=structure,
                    rdp_radius_cache = _rdp_radius_cache,
                    L=regime_spec["L"], stages_per_lag=R_by_l,
                    true_alpha=alpha, true_beta_dict=beta_dict,
                    true_gamma_edge_cov=gamma_edge_cov,
                    true_gamma_time_cov=gamma_time_cov,
                    interaction_pairs=interaction_pairs,
                    true_gamma_interaction=gamma_interaction,
                    T=T, seed=rep,
                )
                result["regime"] = regime_id
                result["structure"] = structure
                all_sim_results.append(result)

density=0.1: radius=0.553, achieved=0.097
density=0.4: radius=1.188, achieved=0.386
Calibrated RDP radius for K=20, density=0.4000: radius=1.188, achieved=0.386


/Users/admin/baboon-gnar-edge/src/model.py:109: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  z_tensor = torch.as_tensor(time_covariates[name], dtype=pred.dtype, device=pred.device).view(-1, 1)  # (N,1)


Displaying results from simulation experiments

In [ ]:


#helper formatting function
def pretty_parameter_label(param):
    if param.startswith("alpha"):
        lag = param.replace("alpha", "")
        return rf"$\alpha_{{{lag}}}$"

    m = re.fullmatch(r"beta(\d+),(\d+)", param)
    if m:
        return rf"$\beta_{{{m.group(1)},{m.group(2)}}}$"

    if param.startswith("gamma_"):
        core = param[len("gamma_") :]

        if "_X_" in core:
            left, right = core.split("_X_", 1)
            left_label = edge_name_map.get(left, left)
            right_label = edge_name_map.get(right, right)
            return rf"$\gamma_{{{left_label}\times {right_label}}}$"

        return rf"$\gamma_{{{edge_name_map.get(core, core)}}}$"

    return param

#building a table for each parameter specification
table_regime = {}

for regime in range(1, 10):
    #excluding results with very high condition numbers for regime 5
    #as this produced some extreme results
    if regime == 5:
        table_regime[regime] = build_per_parameter_table(
            all_sim_results,
            paper_regimes=paper_regimes,
            regime_id=regime,
            T=200,
            exclude_extreme_condition=1000,
        )
    else:
        table_regime[regime] = build_per_parameter_table(
            all_sim_results,
            paper_regimes=paper_regimes,
            regime_id=regime,
            T=200,
        )

#formatting the tables for presentation purposes
formatted_tables = {}

for regime in range(1, 10):
    pretty_table = table_regime[regime].copy()

    pretty_table["parameter"] = pretty_table["parameter"].map(pretty_parameter_label)

    rmse_cols = [c for c in pretty_table.columns if c.endswith("_rmse")]
    coverage_cols = [c for c in pretty_table.columns if c.endswith("_coverage")]

    for col in coverage_cols:
        pretty_table[col] = pretty_table[col].map(
            lambda x: f"{x:.2f}" if pd.notna(x) else ""
        )

    for col in rmse_cols:
        pretty_table[col] = pretty_table[col].map(
            lambda x: f"{x:.1g}" if pd.notna(x) else ""
        )

    formatted_tables[regime] = format_as_multiindex(pretty_table)
    display(formatted_tables[regime])



tables_dir = project_root / "notebooks" / "output" / "tables"

#saving tables in latex form
for regime in range(1, 10):
    df = formatted_tables[regime]

    latex_str = df.to_latex(
        index=True,
        escape=False,
        bold_rows=False,
    )

    caption = f"Coverage rates for 95\% confidence intervals and RMSE for the estimated parameters for simulation regime {regime}."
    label = f"tab:regime_{regime}_params"

    wrapped = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"{latex_str}\n"
        "\\end{table}\n"
    )

    out_file = tables_dir / f"regime_{regime}_table.tex"
    out_file.write_text(wrapped, encoding="utf-8")

    print(f"saved {out_file}")  

Excluded 0 replications with condition_number >= 1000


<>:83: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:83: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
/var/folders/sj/s0ncj4tn3ll32fnddwddmnbw0000gn/T/ipykernel_89224/495958410.py:83: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  caption = f"Coverage rates for 95\% confidence intervals and RMSE for the estimated parameters for simulation regime {regime}."


ER             SBM             RDP       
              Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                    
$\alpha_{1}$      0.96  0.007     0.98  0.008     0.96  0.007
$\beta_{1,1}$     0.96   0.03     0.96   0.03     0.98   0.03

ER             SBM             RDP       
              Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                    
$\alpha_{1}$      0.96  0.007     0.98  0.007     0.98  0.007
$\beta_{1,1}$     0.94   0.03     0.98   0.03     0.96   0.03
$\beta_{1,2}$     0.94   0.04     0.96   0.03     0.98   0.03

ER             SBM             RDP       
              Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                    
$\alpha_{1}$      0.94  0.006     0.92  0.007     0.98  0.006
$\alpha_{2}$      0.94  0.007     0.98  0.006     0.94  0.006
$\alpha_{3}$      0.94  0.007     0.96  0.006     0.94  0.007
$\beta_{1,1}$     0.96   0.02     0.92   0.02     1.00   0.02
$\beta_{2,1}$     1.00   0.02     0.92   0.02     0.94   0.02
$\beta_{3,1}$     0.92   0.02     0.90   0.02     0.92   0.03

...

ER             SBM             RDP       
                          Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                                
$\alpha_{1}$                  1.00  0.005     0.94  0.006     0.98  0.006
$\alpha_{2}$                  0.96  0.006     0.96  0.006     0.94  0.006
$\alpha_{3}$                  0.94  0.007     0.98  0.006     0.92  0.007
$\beta_{1,1}$                 0.98   0.02     0.98   0.02     1.00   0.02
$\beta_{1,2}$                 0.96   0.03     1.00   0.03     0.94   0.03
$\beta_{2,1}$                 0.98   0.02     0.94   0.02     0.96   0.02
$\beta_{2,2}$                 0.96   0.04     0.94   0.03     0.94   0.03
$\beta_{3,1}$                 0.98   0.02     0.96   0.02     0.94   0.02
$\beta_{3,2}$                 0.98   0.03     0.96   0.03     0.96   0.03
$\gamma_{Age difference}$     1.00  0.002     0.98  0.002     1.00  0.002
$\gamma_{Mean age}$           0.96  0.002     0.96  0.002     0.98  0.002

ER             SBM             RDP       
                    Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                          
$\alpha_{1}$            0.98  0.006     1.00  0.005     0.98  0.006
$\alpha_{2}$            0.98  0.005     0.98  0.005     0.96  0.006
$\alpha_{3}$            0.96  0.006     0.98  0.006     0.94  0.007
$\beta_{1,1}$           0.98   0.02     0.98   0.02     0.96   0.02
$\beta_{1,2}$           0.92   0.02     0.96   0.02     0.94   0.02
$\beta_{2,1}$           0.96   0.02     1.00   0.01     0.94   0.02
$\beta_{2,2}$           0.98   0.02     1.00   0.02     0.90   0.02
$\beta_{3,1}$           0.94   0.02     0.96   0.02     0.90   0.02
$\beta_{3,2}$           0.90   0.02     0.98   0.02     0.96   0.02
$\gamma_{sim_temp}$     0.92  0.002     0.96  0.002     0.96  0.002

ER             SBM             RDP       
                          Coverage   RMSE Coverage   RMSE Coverage   RMSE
parameter                                                                
$\alpha_{1}$                  0.90  0.007     0.98  0.006     0.98  0.005
$\alpha_{2}$                  0.94  0.006     0.90  0.006     0.94  0.006
$\alpha_{3}$                  0.96  0.006     1.00  0.006     0.94  0.007
$\beta_{1,1}$                 0.90   0.02     0.96   0.02     0.96   0.02
$\beta_{1,2}$                 0.96   0.02     0.98   0.02     0.94   0.02
$\beta_{2,1}$                 0.96   0.02     0.98   0.02     0.96   0.02
$\beta_{2,2}$                 0.96   0.02     0.98   0.02     0.88   0.02
$\beta_{3,1}$                 0.96   0.02     0.94   0.02     0.92   0.02
$\beta_{3,2}$                 0.96   0.02     0.94   0.02     0.94   0.02
$\gamma_{Age difference}$     0.86  0.002     0.92  0.002     1.00  0.002
$\gamma_{Mean age}$           0.96  0.002     0.96  0.002     0.98  0.002
$\gamma_{sim_temp}$           0.96  0.003     0.94  0.003     0.94  0.002

ER              SBM              RDP  \
                                   Coverage    RMSE Coverage    RMSE Coverage   
parameter                                                                       
$\alpha_{1}$                           0.96   0.006     1.00   0.005     0.96   
$\alpha_{2}$                           0.96   0.005     0.98   0.005     0.96   
$\alpha_{3}$                           0.98   0.006     0.92   0.007     0.94   
$\beta_{1,1}$                          0.96    0.02     0.98    0.01     0.94   
$\beta_{1,2}$                          0.98    0.02     1.00    0.01     0.96   
$\beta_{2,1}$                          0.94    0.01     0.96    0.01     0.98   
$\beta_{2,2}$                          0.98    0.02     0.94    0.01     0.92   
$\beta_{3,1}$                          0.94    0.02     0.94    0.02     0.92   
$\beta_{3,2}$                          0.94    0.02     0.92    0.02     0.96   
$\gamma_{Age difference}$              0.98   0.002     0.98   0.002     0.96   
$\gamma_{Mean age}$                    0.98   0.005     0.94   0.005     0.96   
$\gamma_{sim_temp}$                    0.92   0.005     0.92   0.005     0.96   
$\gamma_{Mean age\times sim_temp}$     0.96  0.0003     0.92  0.0003     0.96   

                                            
                                      RMSE  
parameter                                   
$\alpha_{1}$                         0.005  
$\alpha_{2}$                         0.005  
$\alpha_{3}$                         0.006  
$\beta_{1,1}$                         0.02  
$\beta_{1,2}$                         0.02  
$\beta_{2,1}$                         0.01  
$\beta_{2,2}$                         0.01  
$\beta_{3,1}$                         0.02  
$\beta_{3,2}$                         0.01  
$\gamma_{Age difference}$            0.002  
$\gamma_{Mean age}$                  0.006  
$\gamma_{sim_temp}$                  0.004  
$\gamma_{Mean age\times sim_temp}$  0.0003

saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_1_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_2_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_3_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_4_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_5_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_6_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_7_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_8_table.tex
saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/regime_9_table.tex


***Parameter Estimation Accuracy for networks similar to the real data application***

In [ ]:
#simulation regime used to mirror real data application
regime_large_full = {
    "L": 5,
    "R_max": [1, 1, 1, 1, 1],
    "alpha": [-0.51, -0.35, -0.24, -0.20, -0.10],
    "beta_vals": [0.06, 0.09, 0.26, 0.22, 0.18],
    "gamma_edge_cov": {"age_diff": 0.05, "mean_age": 0.03},
    "gamma_time_cov": 0.2,
    "interaction_pairs": [("mean_age", "sim_temp")],
    "gamma_interaction": {("mean_age", "sim_temp"): 0.02},
}

#resembling number of nodes and density of real interaction network
K_real = 13
density_real = 34 / (13 * 12 / 2)   #~0.436

#converting parameter regime to format expected by simulation functions
R_by_l = expand_R_by_l(regime_large_full["R_max"])
beta_dict = build_beta_dict(R_by_l, regime_large_full["beta_vals"])

n_replications = 50
coverage_results = {}

#looping over structures
for structure in ["ER", "SBM", "RDP"]:
    replication_tables = []
    #repeating n_replications times
    for rep in range(n_replications):
        result_table, learner, graph, diagnostics = run_large_network_replication_density(
            K=K_real, density=density_real, structure=structure,
            rdp_radius_cache = _rdp_radius_cache,
            L=regime_large_full["L"], stages_per_lag=R_by_l,
            true_alpha=regime_large_full["alpha"], true_beta_dict=beta_dict,
            true_gamma_edge_cov=regime_large_full["gamma_edge_cov"],
            true_gamma_time_cov=regime_large_full["gamma_time_cov"],
            interaction_pairs=regime_large_full["interaction_pairs"],
            true_gamma_interaction=regime_large_full["gamma_interaction"],
            T=28, seed=rep,
        )
        replication_tables.append(result_table)

    #building a table to resemble table 4 in gnar-edge paper
    table4_agg = build_aggregated_table4_style(replication_tables, n_replications)
    print(f"\n=== {structure}: Simulation Results (T=28, averaged over {n_replications} replications) ===")
    display(table4_agg)

    #exporting tables to latex
    table4_agg = table4_agg.copy()
    table4_agg["Estimated"] = table4_agg["Estimated"].map(lambda x: f"{x:.2f}")
    table4_agg["95% CI"] = table4_agg["95% CI"].astype(str)
    table4_agg["Coverage"] = table4_agg["Coverage"].astype(str)

    out_file = tables_dir / f"baboon_accuracy_{structure.lower()}.tex"
    table4_agg.to_latex(
        out_file,
        index=False,
        escape=False,
        caption=f"Average estimated parameters, average confidence intervals, and coverage rates over 50 repetitions for {structure} networks similar to the baboon network.",
        label=f"tab:baboon_accuracy_{structure.lower()}",
        column_format="l l l l",
    )

    print(f"saved {out_file}")


=== ER: Simulation Results (T=28, averaged over 50 replications) ===


,parameter,Estimated,95% CI,Coverage
0,age_diff,0.049015,"(0.031, 0.067)",98%
1,alpha1,-0.513123,"(-0.575, -0.451)",96%
2,alpha2,-0.356047,"(-0.421, -0.291)",94%
3,alpha3,-0.246310,"(-0.311, -0.182)",94%
4,alpha4,-0.203291,"(-0.266, -0.140)",100%
5,alpha5,-0.102081,"(-0.162, -0.042)",94%
6,"beta1,1",0.061180,"(-0.006, 0.128)",96%
7,"beta2,1",0.093071,"(0.024, 0.162)",88%
8,"beta3,1",0.267052,"(0.199, 0.335)",98%
9,"beta4,1",0.220770,"(0.154, 0.287)",98%


saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/baboon_accuracy_er.tex

=== SBM: Simulation Results (T=28, averaged over 50 replications) ===


,parameter,Estimated,95% CI,Coverage
0,age_diff,0.046944,"(0.029, 0.065)",96%
1,alpha1,-0.513046,"(-0.575, -0.451)",96%
2,alpha2,-0.356790,"(-0.421, -0.292)",94%
3,alpha3,-0.240072,"(-0.304, -0.176)",96%
4,alpha4,-0.200338,"(-0.263, -0.138)",92%
5,alpha5,-0.099323,"(-0.159, -0.039)",98%
6,"beta1,1",0.064594,"(-0.002, 0.131)",98%
7,"beta2,1",0.096158,"(0.028, 0.165)",96%
8,"beta3,1",0.260621,"(0.193, 0.328)",98%
9,"beta4,1",0.219069,"(0.153, 0.285)",92%


saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/baboon_accuracy_sbm.tex

=== RDP: Simulation Results (T=28, averaged over 50 replications) ===


,parameter,Estimated,95% CI,Coverage
0,age_diff,0.049014,"(0.032, 0.066)",96%
1,alpha1,-0.513918,"(-0.575, -0.452)",94%
2,alpha2,-0.355111,"(-0.419, -0.291)",98%
3,alpha3,-0.235689,"(-0.299, -0.173)",96%
4,alpha4,-0.205513,"(-0.267, -0.144)",94%
5,alpha5,-0.097308,"(-0.157, -0.038)",92%
6,"beta1,1",0.061687,"(-0.005, 0.128)",94%
7,"beta2,1",0.091882,"(0.024, 0.159)",96%
8,"beta3,1",0.258094,"(0.191, 0.325)",96%
9,"beta4,1",0.225176,"(0.160, 0.291)",94%


saved /Users/admin/baboon-gnar-edge/notebooks/output/tables/baboon_accuracy_rdp.tex
